In [1]:
import numpy as np
import pandas as pd
import zipfile

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv("spambase.data", header=None)

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values.reshape(-1, 1)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def add_intercept(X):
    return np.c_[np.ones((X.shape[0], 1)), X]

X_train_b = add_intercept(X_train_scaled)
X_test_b = add_intercept(X_test_scaled)

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def cross_entropy_loss(X, y, theta):
    p = sigmoid(X @ theta)
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def logistic_gradient_descent(X, y, alpha, num_iters):
    n, d = X.shape
    theta = np.zeros((d, 1))
    losses = []

    for _ in range(num_iters):
        p = sigmoid(X @ theta)
        grad = (X.T @ (p - y)) / n
        theta = theta - alpha * grad
        losses.append(cross_entropy_loss(X, y, theta))

    return theta, losses

def predict_labels(X, theta, threshold=0.5):
    probs = sigmoid(X @ theta)
    return (probs >= threshold).astype(int)

In [5]:
alphas = [0.01, 0.1, 0.5]
iters_to_report = [10, 50, 100]

loss_rows = []
metric_rows = []

for alpha in alphas:
    theta, losses = logistic_gradient_descent(X_train_b, y_train, alpha, 100)

    loss_rows.append([
        alpha,
        losses[9],
        losses[49],
        losses[99]
    ])

    y_pred_train = predict_labels(X_train_b, theta)
    y_pred_test = predict_labels(X_test_b, theta)

    metric_rows.append([
        alpha,
        accuracy_score(y_train, y_pred_train),
        precision_score(y_train, y_pred_train),
        recall_score(y_train, y_pred_train),
        f1_score(y_train, y_pred_train),
        accuracy_score(y_test, y_pred_test),
        precision_score(y_test, y_pred_test),
        recall_score(y_test, y_pred_test),
        f1_score(y_test, y_pred_test)
    ])

In [6]:
loss_df = pd.DataFrame(
    loss_rows,
    columns=["Learning Rate", "Loss @ 10", "Loss @ 50", "Loss @ 100"]
)

metrics_df = pd.DataFrame(
    metric_rows,
    columns=[
        "Learning Rate",
        "Train Accuracy", "Train Precision", "Train Recall", "Train F1",
        "Test Accuracy", "Test Precision", "Test Recall", "Test F1"
    ]
)

print("Gradient Descent Loss Results:")
print(loss_df)
print()

print("Gradient Descent Metrics at 100 Iterations:")
print(metrics_df)
print()

pkg_model = LogisticRegression(max_iter=1000)
pkg_model.fit(X_train_scaled, y_train.ravel())

pkg_train_pred = pkg_model.predict(X_train_scaled)
pkg_test_pred = pkg_model.predict(X_test_scaled)

print("Package Logistic Regression Metrics:")
print("Training Set:")
print("Accuracy:", accuracy_score(y_train, pkg_train_pred))
print("Precision:", precision_score(y_train, pkg_train_pred))
print("Recall:", recall_score(y_train, pkg_train_pred))
print("F1:", f1_score(y_train, pkg_train_pred))
print()

print("Testing Set:")
print("Accuracy:", accuracy_score(y_test, pkg_test_pred))
print("Precision:", precision_score(y_test, pkg_test_pred))
print("Recall:", recall_score(y_test, pkg_test_pred))
print("F1:", f1_score(y_test, pkg_test_pred))

Gradient Descent Loss Results:
   Learning Rate  Loss @ 10  Loss @ 50  Loss @ 100
0           0.01   0.651167   0.541620    0.468734
1           0.10   0.465377   0.324124    0.289078
2           0.50   0.320251   0.258894    0.243778

Gradient Descent Metrics at 100 Iterations:
   Learning Rate  Train Accuracy  Train Precision  Train Recall  Train F1  \
0           0.01        0.897681         0.873389      0.860987  0.867143   
1           0.10        0.907826         0.911290      0.844544  0.876649   
2           0.50        0.918261         0.923756      0.860239  0.890867   

   Test Accuracy  Test Precision  Test Recall   Test F1  
0       0.903562        0.900881     0.861053  0.880517  
1       0.906169        0.923788     0.842105  0.881057  
2       0.915725        0.939535     0.850526  0.892818  

Package Logistic Regression Metrics:
Training Set:
Accuracy: 0.9257971014492754
Precision: 0.9239811912225705
Recall: 0.8811659192825112
F1: 0.9020657995409335

Testing Set:
Accu